In [1]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

In [8]:
# resize function
def resize(image, maximum_s = 700, devided_by = 3):
    h, w = image.shape[0], image.shape[1]
    if h > maximum_s or w > maximum_s:
        if h > w*1.5:
            image = cv.resize(image, (w // devided_by, h // (devided_by+1)), interpolation=cv.INTER_AREA)
        else:
            image = cv.resize(image, (w // devided_by, h // devided_by), interpolation=cv.INTER_AREA)
    return image


In [9]:
cap = cv.VideoCapture("data\street1.mp4")

# fps = cap.get(cv.CAP_PROP_FPS)
# width = 720
# height = 960
# fourcc = cv.VideoWriter_fourcc(*'mp4v')
# out = cv.VideoWriter("data\output.mp4",
#                     fourcc,
#                     fps // 5,
#                     (width, height))



vehicles = 0
frame_ignore = 0
car = False

while True:
    # frame jumping for getting better quality and speed.
    rec, f_jump = cap.read()
    rec, f_jump = cap.read()
    rec, f_jump = cap.read()
    

    rec, frame1 = cap.read()
    rec, frame2 = cap.read()
    if not rec:
        break

    # rejectiong some frames for speed and better result
    if frame_ignore == 1:
        frame_ignore = 0
        continue
    else:
        frame_ignore += 1

    # resizing
    frame1 = resize(frame1)
    frame2 = resize(frame2)

    
    # seprating zoom section. this coordinates are customized.
    focus1 = frame1[720:955, 0:715].copy()
    focus2 = frame2[720:955, 0:715].copy()

    # detecting defferences
    frame_diff = cv.absdiff(focus1, focus2)

    # preprocessing
    gray = cv.cvtColor(frame_diff, cv.COLOR_BGR2GRAY)
    _ , thresh = cv.threshold(gray, 10, 255, cv.THRESH_BINARY)
    dilated = cv.dilate(thresh, (13,13), iterations=1)
    closed = cv.morphologyEx(dilated, cv.MORPH_CLOSE, (9,9))
    blured = cv.GaussianBlur(closed, (9,9), 1)
    eroded = cv.erode(blured, (10,10), iterations=1)

    # making contour and showing vehicles
    contours, _ = cv.findContours(eroded, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    
    for c in contours:
        if cv.contourArea(c) > 1300:
            x,y,w,h = cv.boundingRect(c)
            cv.rectangle(focus2, (x,y), (x+w,y+h), (0,0,255), 3)
            

    # counting the vehicles in the specific section.
    section = eroded[0:40, 0:715].copy()
    check_section_contours, _ = cv.findContours(section, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    sorted_contours = sorted(check_section_contours, key=cv.contourArea, reverse=True)[:1]

    for c in sorted_contours:
        if cv.contourArea(c) > 1300:
            x,y,w,h = cv.boundingRect(c)

            if w > 100 and car == True:
                continue
            elif w > 100 and car == False:
                vehicles += 1
                car = True
            else:
                vehicles += 1
            

                
                
    # adapt
    frame2[720:955, 0:715] = focus2


    
    cv.rectangle(frame2, (0,720), (715,955), (0,0,255), 3)
    cv.line(frame2, (0,760), (715,760), (0,255,0), 3)
    cv.putText(frame2, f"vehicles: {vehicles}", (10,30), cv.FONT_HERSHEY_COMPLEX, 1, (0,0,255), 1)


    cv.imshow("frame", frame2)

    # out.write(frame2)
    if cv.waitKey(1) & 0xFF == 27:
        break


cv.destroyAllWindows()
cap.release()
# out.release()